# Day 2 Lab: Causal & Fair Modelling for Credit Decisions
We will blend Bloomberg spreads with macro indicators (or synthetic stand-ins) to estimate treatment effects and audit fairness.

> **Learning outcomes**
> - engineer panel data joining `CRPR <GO>` spread exports with macro factors
> - estimate treatment effects with Double Machine Learning (DML)
> - compute demographic parity & equalized odds metrics
> - package findings into a model risk memo

## 0. Runtime prep
- Ensure **GPU is off** (CPU is fine, but enable High-RAM if available).
- Upload Bloomberg exports: `credit_spreads_YYYYMMDD.csv` (from `CRPR <GO>` or `CACS <GO>`) and `macro_indicators.csv` (from `ECST <GO>` or `WECO <GO>`).
- If data is unavailable, the notebook will generate a synthetic sample so you can still complete the exercises.

In [ ]:
%%capture
!pip install pandas numpy scikit-learn statsmodels plotly==5.24.0 econml==0.15.1 fairlearn shap

In [ ]:
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except ImportError:
    drive = None
    IN_COLAB = False

DATA_ROOT = Path("/content/data") if IN_COLAB else Path.cwd() / "data" / "bloomberg"
DATA_ROOT.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    drive.mount("/content/drive", force_remount=True)


### Data expectations
- `credit_spreads_YYYYMMDD.csv` columns: `ticker`, `asof_date`, `rating`, `sector`, `region`, `spread_bps`.
- `macro_indicators.csv` columns: `asof_date`, `gdp_surprise`, `inflation_surprise`, `policy_rate`, `sentiment_score`.
- Optional fairness attribute join: use `issuer_country` → map to `protected_group` (e.g., Emerging vs Developed) or load your own compliance dimension.

In [ ]:
spreads_path = DATA_ROOT / "credit_spreads_SAMPLE.csv"  # rename!
macro_path = DATA_ROOT / "macro_indicators_SAMPLE.csv"

try:
    spreads = pd.read_csv(spreads_path, parse_dates=["asof_date"])
    macro = pd.read_csv(macro_path, parse_dates=["asof_date"])
    print("Loaded Bloomberg exports.")
except FileNotFoundError:
    print("Bloomberg exports not found. Generating synthetic panel...")
    rng = np.random.default_rng(42)
    dates = pd.date_range("2019-01-31", periods=36, freq="M")
    tickers = [f"Corp{i:02d}" for i in range(30)]
    records = []
    for date in dates:
        for ticker in tickers:
            sector = rng.choice(["Financials", "Energy", "Healthcare", "Tech"])
            region = rng.choice(["DM", "EM"])
            rating = rng.choice(["AAA", "AA", "A", "BBB"])
            spread = rng.normal(120, 40)
            records.append({
                "ticker": ticker,
                "asof_date": date,
                "rating": rating,
                "sector": sector,
                "region": region,
                "spread_bps": max(40, spread)
            })
    spreads = pd.DataFrame(records)
    macro = pd.DataFrame({
        "asof_date": dates,
        "gdp_surprise": rng.normal(0, 0.5, len(dates)),
        "inflation_surprise": rng.normal(0, 0.3, len(dates)),
        "policy_rate": rng.normal(1.5, 0.25, len(dates)).cumsum(),
        "sentiment_score": rng.uniform(-1, 1, len(dates))
    })

def engineer_protected(df: pd.DataFrame) -> pd.Series:
    return df["region"].map({"DM": "developed", "EM": "emerging"})

spreads["protected_group"] = engineer_protected(spreads)
data = spreads.merge(macro, on="asof_date", how="left")
data.head()

In [ ]:
data = data.assign(
    treatment=lambda d: (d["sentiment_score"] < -0.3).astype(int),
    outcome=lambda d: d["spread_bps"],
    risk_score=lambda d: d["spread_bps"] + 0.5 * d["inflation_surprise"] * 100,
)
feature_cols = [
    "rating", "sector", "region", "gdp_surprise", "inflation_surprise",
    "policy_rate", "sentiment_score"
]
X = pd.get_dummies(data[feature_cols], drop_first=True)
T = data["treatment"].values
Y = data["outcome"].values
protected = data["protected_group"].values

In [ ]:
from econml.dml import LinearDML
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

model_y = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42)
model_t = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)

linear_dml = LinearDML(
    model_y=model_y,
    model_t=model_t,
    discrete_treatment=True,
    cv=3,
    random_state=42
)
linear_dml.fit(Y, T, X=X)
ate = linear_dml.ate(X)
print(f"Estimated Average Treatment Effect (bps): {ate:.2f}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.ensemble import GradientBoostingClassifier
from fairlearn.metrics import MetricFrame, selection_rate, false_positive_rate, false_negative_rate

risk_flag = (data["spread_bps"] > data["spread_bps"].median()).astype(int)
X_train, X_test, y_train, y_test, prot_train, prot_test = train_test_split(
    X, risk_flag, protected, test_size=0.3, random_state=42, stratify=risk_flag
)
clf = GradientBoostingClassifier(random_state=42)
clf.fit(X_train, y_train)
probs = clf.predict_proba(X_test)[:, 1]
preds = (probs > 0.5).astype(int)

print("ROC AUC:", roc_auc_score(y_test, probs))
metrics = {
    "selection_rate": selection_rate,
    "fpr": false_positive_rate,
    "fnr": false_negative_rate
}
metric_frame = MetricFrame(metrics=metrics, y_true=y_test, y_pred=preds, sensitive_features=prot_test)
metric_frame.by_group

In [ ]:
disparity = metric_frame.by_group.max() - metric_frame.by_group.min()
disparity

### Debiasing exercise
Use the template below to try one mitigation technique:
- reweighing sensitive groups
- threshold adjustment per group
- post-model calibration (e.g., equalized odds via `fairlearn.reductions`).

In [ ]:
from fairlearn.reductions import ExponentiatedGradient, EqualizedOdds
from sklearn.linear_model import LogisticRegression

mitigator = ExponentiatedGradient(
    estimator=LogisticRegression(max_iter=200),
    constraints=EqualizedOdds()
)
mitigator.fit(X_train, y_train, sensitive_features=prot_train)
mitigated_preds = mitigator.predict(X_test)
MetricFrame(metrics=metrics, y_true=y_test, y_pred=mitigated_preds, sensitive_features=prot_test).by_group

### Deliverable
Draft a one-page memo covering:
1. Data provenance & any assumptions made when substituting Bloomberg values.
2. Estimated treatment effect and its interpretation for a credit committee.
3. Fairness diagnostics (metric values + mitigation technique) with next steps.
Upload the memo plus this notebook to the LMS / assessment portal.